In [1]:
#!pip install -U pycocotools

In [2]:
import json
from pathlib import Path
from collections import defaultdict

import evaluate

ROOT = Path("/lambda/nfs/neel/Research")
PRED_PATH = ROOT / "runs" / "dinov2" / "coco_caption_5k" / "preds_dinov2_vitb14.jsonl"
GT_META  = ROOT / "subsets" / "coco_caption_5k" / "metadata.jsonl"
OUT_DIR  = ROOT / "runs" / "dinov2" / "coco_caption_5k" / "eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def read_jsonl(p: Path):
    rows = []
    with p.open("r", encoding="utf-8") as f:
        for line in f:
            line=line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def norm_caps(x):
    if isinstance(x, str): return [x]
    if isinstance(x, list):
        out=[]
        for t in x:
            if isinstance(t, str): out.append(t)
            elif isinstance(t, dict) and "caption" in t: out.append(str(t["caption"]))
            else: out.append(str(t))
        return out
    if isinstance(x, dict):
        for k in ["captions","caption","references","texts"]:
            if k in x: return norm_caps(x[k])
    return [str(x)]

preds = read_jsonl(PRED_PATH)
gt_rows = read_jsonl(GT_META)
sample = gt_rows[0]
cap_key = next(k for k in ["captions","caption","references","texts"] if k in sample)

gt_by_id = {}
for i, r in enumerate(gt_rows):
    iid = r.get("image_id", r.get("id", i))
    gt_by_id[iid] = norm_caps(r[cap_key])

# align refs for each prediction (multi-ref per image)
y_pred = []
y_refs = []
missing = 0
for r in preds:
    iid = r["image_id"]
    refs = gt_by_id.get(iid, gt_by_id.get(int(iid) if str(iid).isdigit() else iid, None))
    if not refs:
        missing += 1
        continue
    y_pred.append(r.get("pred_caption",""))
    y_refs.append(refs)

print("n_pred:", len(preds), "n_scored:", len(y_pred), "missing_refs:", missing)

bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")
meteor = evaluate.load("meteor")

# BLEU expects references as List[List[str]]
bleu_out = bleu.compute(predictions=y_pred, references=y_refs)

# ROUGE expects references as List[str]; use first ref for rouge (common practice)
rouge_out = rouge.compute(predictions=y_pred, references=[refs[0] for refs in y_refs])

# METEOR supports multi-ref if you pass list-of-lists (works with evaluate meteor)
meteor_out = meteor.compute(predictions=y_pred, references=y_refs)

metrics = {
    "BLEU": float(bleu_out["bleu"]),
    "ROUGE1": float(rouge_out["rouge1"]),
    "ROUGE2": float(rouge_out["rouge2"]),
    "ROUGEL": float(rouge_out["rougeL"]),
    "METEOR": float(meteor_out["meteor"]),
    "n_scored": len(y_pred),
}

out_path = OUT_DIR / "metrics_dinov2_caption_py3.json"
out_path.write_text(json.dumps(metrics, indent=2), encoding="utf-8")
print(metrics)
print("saved:", out_path)


/home/ubuntu/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


n_pred: 5000 n_scored: 5000 missing_refs: 0


[nltk_data] Downloading package wordnet to /home/ubuntu/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/ubuntu/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package omw-1.4 to /home/ubuntu/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


{'BLEU': 0.11810632421029793, 'ROUGE1': 0.2918864321300437, 'ROUGE2': 0.07262514078049717, 'ROUGEL': 0.2630551421638991, 'METEOR': 0.35511630027089186, 'n_scored': 5000}
saved: /lambda/nfs/neel/Research/runs/dinov2/coco_caption_5k/eval/metrics_dinov2_caption_py3.json
